# inplace-op-unsafe-warning — faded example 1: Complete the recipe guard in sub_inplace_safe

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `inplace-op-unsafe-warning`. Running the beacon reports progress on the `Backprop: In-place op unsafe warning` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: In-place op unsafe warning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-op-unsafe-warning`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-op-unsafe-warning"
DD_SUBTOPIC = "Backprop: In-place op unsafe warning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

An in-place op overwrites a tensor's `.array`, which is unsafe if that array is cached as a parent in some downstream Recipe. The signal for 'cached somewhere' is `x.recipe is not None` (non-leaf). Leaves carry no Recipe and are safe to mutate.

## Faded exercise 1

Complete `sub_inplace_safe(x, y)`, an in-place `x -= y` that protects the graph. It must `raise RuntimeError` (message containing 'in-place') when `x` is a non-leaf, and otherwise mutate `x.array` and return `x`. Fill in the guard condition.

**Fill in:** the condition that detects a graph-intermediate (non-leaf) tensor

In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.recipe = recipe

def sub_inplace_safe(x, y):
    is_unsafe = None  # TODO: the condition that detects a graph-intermediate (non-leaf) tensor
    if is_unsafe:
        raise RuntimeError('in-place op forbidden on a Tensor with a recipe')
    x.array -= y.array
    return x


def _test():
    # leaf path: mutates, returns same object
    leaf = MiniTensor([10.0, 20.0])
    res = sub_inplace_safe(leaf, MiniTensor([1.0, 2.0]))
    assert res is leaf
    assert np.allclose(leaf.array, [9.0, 18.0])
    # non-leaf path: must raise with 'in-place' in message
    node = MiniTensor([5.0, 5.0], recipe=Recipe(np.add, (), {}, {}))
    raised = False
    try:
        sub_inplace_safe(node, MiniTensor([1.0, 1.0]))
    except RuntimeError as e:
        raised = True
        assert 'in-place' in str(e)
    assert raised, 'guard must fire on a recipe-carrying tensor'
    # node must be untouched after the refusal
    assert np.allclose(node.array, [5.0, 5.0])


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.recipe = recipe

def sub_inplace_safe(x, y):
    is_unsafe = x.recipe is not None
    if is_unsafe:
        raise RuntimeError('in-place op forbidden on a Tensor with a recipe')
    x.array -= y.array
    return x
```
</details>